In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

df=pd.read_csv('Project_description_and_data/claims_train.csv')

df['VehAge_1']=df['VehAge'] +1
df['VehAge_log']=np.log(df['VehAge_1'])

df['DrivAge_log']=np.log(df['DrivAge'])
df['Density_log']=np.log(df['Density'])

low_cn=df[df['ClaimNb']>1]

In [ ]:
low_cn['VehGas']=np.where((low_cn['VehGas']=='Regular'), 0, 1)

features = ['ClaimNb', 'DrivAge_log', 'VehPower', 'VehAge_log', 'Density_log', 'BonusMalus']

from sklearn.preprocessing import StandardScaler

X = low_cn[features].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

from sklearn.decomposition import PCA

pca = PCA(n_components=2)   # for 2D visualization
X_pca = pca.fit_transform(X_scaled)

low_cn["PC1"] = X_pca[:, 0]
low_cn["PC2"] = X_pca[:, 1]

In [ ]:
sns.scatterplot(
    x="PC1",
    y="PC2",
    hue="ClaimNb",
    data=low_cn,
    palette="viridis",
    alpha=0.7
)

#plt.ylim(-3, 5)
plt.title("PCA Projection Colored by Claim Number")
plt.show()

In [ ]:
explained = pca.explained_variance_ratio_
print(f"PC1: {explained[0]:.2%}, PC2: {explained[1]:.2%}, Total: {explained[:2].sum():.2%}")

loadings = pd.DataFrame(
    pca.components_.T,
    columns=["PC1", "PC2"],
    index=features
)
print(loadings)

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=7, random_state=42)
low_cn["Cluster"] = kmeans.fit_predict(X_pca)

sns.scatterplot(x="PC1", y="PC2", hue="Cluster", data=low_cn, palette="tab10")
plt.title("PCA + KMeans Clustering")
plt.show()